In [0]:
%run ./00_Configuration

## 1. Parameters and storage paths

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType
from delta.tables import DeltaTable
import csv

# Existing 00_Configuration defines input_path and bronze_path.
raw_path = input_path.rstrip("/") + "/"
bronze_data_path = bronze_path.rstrip("/") + "/data"
registry_path = bronze_path.rstrip("/") + "/file_registry"
metadata_path = bronze_path.rstrip("/") + "/file_metadata"

# Safe default: incremental. Use full only when intentionally rebuilding Bronze.
dbutils.widgets.dropdown("ingestion_mode", "incremental", ["incremental", "full"], "Ingestion Mode")
ingestion_mode = dbutils.widgets.get("ingestion_mode").strip().lower()
if ingestion_mode not in {"incremental", "full"}:
    raise ValueError(f"Unsupported ingestion_mode: {ingestion_mode}")

metadata_columns = ["source_file", "ingestion_timestamp", "pipeline_name"]
time_columns = ["sch_arrival", "sch_departure", "act_arrival", "act_departure"]
pipeline_name = "railpulse_india"
print("Ingestion mode:", ingestion_mode)
print("Raw path:", raw_path)
print("Bronze path:", bronze_data_path)

## 2. Discover files and load the expected schema

In [0]:
# Directory listing is deliberately scoped to the existing raw CSV directory.
raw_files = sorted(
    [x for x in dbutils.fs.ls(raw_path) if x.name.lower().endswith(".csv")],
    key=lambda x: x.path
)
if not raw_files:
    raise ValueError(f"No CSV files found at {raw_path}")

raw_info = {x.path: {"name": x.name, "size": int(x.size),
                     "modified": int(x.modificationTime) if x.modificationTime is not None else None}
            for x in raw_files}

# Check whether a Delta dataset is readable; don't swallow unrelated failures.
def delta_exists(path):
    try:
        return DeltaTable.isDeltaTable(spark, path)
    except Exception as exc:
        raise RuntimeError(f"Cannot check Delta dataset at {path}") from exc

bronze_exists = delta_exists(bronze_data_path)
registry_exists = delta_exists(registry_path)
snapshot_exists = delta_exists(metadata_path)

if bronze_exists:
    existing_bronze = spark.read.format("delta").load(bronze_data_path)
    expected_columns = [c for c in existing_bronze.columns if c not in metadata_columns]
    business_schema = existing_bronze.select(*expected_columns).schema
else:
    if ingestion_mode == "incremental":
        raise RuntimeError("Bronze does not exist. Run ingestion_mode='full' once to initialize it.")
    # First-time bootstrap only: infer schema from the full set of original CSV files.
    bootstrap_df = (spark.read.option("header", True).option("inferSchema", True)
                    .csv([x.path for x in raw_files]))
    expected_columns = bootstrap_df.columns
    for field in bootstrap_df.schema.fields:
        if field.dataType.typeName().startswith("time"):
            bootstrap_df = bootstrap_df.withColumn(field.name, F.col(field.name).cast("string"))
    business_schema = bootstrap_df.schema

if len(expected_columns) != len(set(expected_columns)):
    raise ValueError("Duplicate business column names detected")

print("Raw CSV files:", len(raw_files))
print("Expected business columns:", len(expected_columns))
print("Bronze already exists:", bronze_exists)
print("Registry already exists:", registry_exists)
print("Metadata snapshot already exists:", snapshot_exists)

## 3. Find files needing ingestion

In [0]:
# The existing registry records processed file paths; the snapshot records
# file size and modification time. A missing snapshot entry triggers reprocessing.
processed = set()
if registry_exists:
    processed = {r["source_file"] for r in
                 (spark.read.format("delta").load(registry_path)
                  .filter(F.col("status") == "PROCESSED")
                  .select("source_file").distinct().collect())}

snapshot = {}
if snapshot_exists:
    snapshot = {r["source_file"]: (r["file_size_bytes"], r["modified_at_ms"])
                for r in (spark.read.format("delta").load(metadata_path)
                          .select("source_file", "file_size_bytes", "modified_at_ms").collect())}

new_files = [p for p in raw_info if p not in processed]
modified_files = [p for p in raw_info if p in processed and
                  (p not in snapshot or
                   (raw_info[p]["size"], raw_info[p]["modified"]) != snapshot[p])]
files_to_ingest = (sorted(raw_info) if ingestion_mode == "full"
                   else sorted(set(new_files + modified_files)))

print("New files:", len(new_files))
print("Modified/missing-snapshot files:", len(modified_files))
print("Files selected for this run:", len(files_to_ingest))
for path in files_to_ingest:
    print("  ", path)
if not files_to_ingest:
    print("SKIPPED: No new or changed files. Existing Bronze is untouched.")

## 4. Validate CSV headers and prepare rows

In [0]:
# Spark's text reader works on Databricks Serverless (no sparkContext needed).
# FAILFAST ensures malformed typed CSV values fail instead of silently becoming null.
def read_validated_csv(path):
    first_line = spark.read.text(path).select("value").first()
    if first_line is None:
        raise ValueError(f"Empty CSV: {path}")
    actual = next(csv.reader([first_line["value"].lstrip("\ufeff")]))
    if actual != expected_columns:
        raise ValueError(f"CSV header mismatch in {path}. Expected {expected_columns}, got {actual}")

    raw = (spark.read.option("header", True)
           .option("enforceSchema", "false")
           .option("mode", "FAILFAST")
           .schema(business_schema)
           .csv(path))
    # Resolve source path from the actual Spark file metadata, not a hard-coded string.
    prepared = (raw.withColumn("source_file", F.col("_metadata.file_path"))
                .withColumn("ingestion_timestamp", F.current_timestamp())
                .withColumn("pipeline_name", F.lit(pipeline_name)))
    # Align precisely with the Bronze column order, including on initial bootstrap.
    output_cols = (existing_bronze.columns if bronze_exists
                   else expected_columns + metadata_columns)
    prepared = prepared.select(*output_cols)
    # Materialize before write: catches parsing failures and empty files.
    count = prepared.count()
    if count == 0:
        raise ValueError(f"CSV contains no data rows: {path}")
    actual_sources = [r["source_file"] for r in prepared.select("source_file").distinct().collect()]
    if len(actual_sources) != 1 or actual_sources[0].rstrip("/") != path.rstrip("/"):
        raise ValueError(f"Unexpected source_file metadata for {path}: {actual_sources}")
    return prepared, count

print("CSV validation helper ready. No data has been written yet.")

## 5. Write Bronze and update file tracking

In [0]:
# Each file is written with an atomic Delta replaceWhere transaction.
# Replaying the same file replaces only its previous rows, never duplicates them.
# Tracking is updated *after* the Bronze transaction. A retry after partial
# tracking failure is safe because replaceWhere is idempotent for that file.

def tracking_frames(path, count):
    info = raw_info[path]
    registry_row = spark.createDataFrame(
        [(path, int(count), "PROCESSED")],
        "source_file string, record_count long, status string"
    ).withColumn("first_ingested_at", F.current_timestamp())
    metadata_row = spark.createDataFrame(
        [(path, info["name"], info["size"], info["modified"])],
        "source_file string, file_name string, file_size_bytes long, modified_at_ms long"
    ).withColumn("snapshot_timestamp", F.current_timestamp())
    return registry_row, metadata_row


def upsert_tracking(path, count):
    reg, meta = tracking_frames(path, count)
    if delta_exists(registry_path):
        (DeltaTable.forPath(spark, registry_path).alias("t")
         .merge(reg.alias("s"), "t.source_file = s.source_file")
         .whenMatchedUpdate(set={"record_count": "s.record_count",
                                 "status": "s.status",
                                 "first_ingested_at": "s.first_ingested_at"})
         .whenNotMatchedInsertAll().execute())
    else:
        reg.write.format("delta").mode("overwrite").save(registry_path)
    if delta_exists(metadata_path):
        (DeltaTable.forPath(spark, metadata_path).alias("t")
         .merge(meta.alias("s"), "t.source_file = s.source_file")
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    else:
        meta.write.format("delta").mode("overwrite").save(metadata_path)

if ingestion_mode == "full":
    # Validate all files first; avoid clearing Bronze if a file is malformed.
    validated = []
    for path in files_to_ingest:
        part, cnt = read_validated_csv(path)
        validated.append((path, part, cnt))
        print("Validated:", raw_info[path]["name"], "rows:", cnt)

    # Full refresh is intentional; overwrite the complete Bronze dataset once.
    from functools import reduce
    full_df = reduce(lambda a, b: a.unionByName(b), [x[1] for x in validated])
    full_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_data_path)
    print("Full Bronze refresh completed.")

    # Rebuild tracking after successful Bronze commit, so removed raw files
    # cannot linger in the registry after an intentional full refresh.
    registry_rows = [(p, int(cnt), "PROCESSED") for p, _, cnt in validated]
    reg_df = (spark.createDataFrame(registry_rows,
              "source_file string, record_count long, status string")
              .withColumn("first_ingested_at", F.current_timestamp()))
    meta_rows = [(p, raw_info[p]["name"], raw_info[p]["size"], raw_info[p]["modified"])
                 for p, _, _ in validated]
    meta_df = (spark.createDataFrame(meta_rows,
               "source_file string, file_name string, file_size_bytes long, modified_at_ms long")
               .withColumn("snapshot_timestamp", F.current_timestamp()))
    reg_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(registry_path)
    meta_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(metadata_path)
    print("Registry and metadata snapshot rebuilt.")

else:
    for path in files_to_ingest:
        prepared, count = read_validated_csv(path)
        # Escape a single quote in a path for the SQL replaceWhere predicate.
        escaped_path = path.replace("'", "''")
        (prepared.write.format("delta")
         .mode("overwrite")
         .option("replaceWhere", f"source_file = '{escaped_path}'")
         .save(bronze_data_path))
        print("Bronze committed:", raw_info[path]["name"], "rows:", count)
        upsert_tracking(path, count)
        print("Tracking committed:", raw_info[path]["name"])

print("INGESTION COMPLETE:", ingestion_mode)

## 6. Verify persisted Bronze and tracking

In [0]:
bronze_verify_df = spark.read.format("delta").load(bronze_data_path)
registry_verify_df = spark.read.format("delta").load(registry_path)
metadata_verify_df = spark.read.format("delta").load(metadata_path)

bronze_count = bronze_verify_df.count()
registry_count = registry_verify_df.filter(F.col("status") == "PROCESSED").count()
metadata_count = metadata_verify_df.count()
print("Bronze records:", bronze_count)
print("Bronze columns:", len(bronze_verify_df.columns))
print("Processed registry files:", registry_count)
print("Metadata snapshot files:", metadata_count)

# Cross-check the registry against the actual rows by source file.
bronze_by_file = bronze_verify_df.groupBy("source_file").agg(F.count("*").alias("actual_count"))
registry_by_file = (registry_verify_df.filter(F.col("status") == "PROCESSED")
                    .select("source_file", F.col("record_count").alias("registered_count")))
mismatches = (bronze_by_file.join(registry_by_file, "source_file", "full_outer")
              .filter(F.col("actual_count").isNull() |
                      F.col("registered_count").isNull() |
                      (F.col("actual_count") != F.col("registered_count"))))
if mismatches.limit(1).count() > 0:
    display(mismatches)
    raise RuntimeError("Bronze and registry record counts disagree")
print("PASS: Bronze and registry record counts match by file.")
display(bronze_by_file.orderBy("source_file"))